In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import FunctionTransformer,StandardScaler,OneHotEncoder,OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score,GridSearchCV
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score,mean_absolute_error,root_mean_squared_error,mean_squared_error
from sklearn.ensemble import VotingRegressor

In [2]:
df = pd.read_csv('cleaned_engineered.csv')

In [8]:
skew_num = ['Study_Hours']
other_num = ['Age','Avg_Daily_Usage_Hours','Daily_Unlocks','Physical_Activity_Hours','Sleep_Hours_Per_Night']
ord_cat = ['Stress_Level','Academic_Level']
ohe_cat = ['Gender','Country','Most_Used_Platform','Purpose_Of_Use']
cols = skew_num+other_num+ord_cat+ohe_cat
X = df[cols]
y = df['Mental_Health_Score']

In [9]:
#1. Skewed features
skew_pipeline = Pipeline(steps=[
    ('log_transform', FunctionTransformer(np.log1p)),
    ('scale', StandardScaler())

])

#2. Numeric Features
plain_numeric_pipeline = Pipeline(steps=[
    ('scale',StandardScaler())
])

#3. Ordinal
ordinal_pipeline = Pipeline(steps=[
    ('encode', OrdinalEncoder(categories=[['Low', 'Medium', 'High', 'Very High'],['High School','Undergraduate','Graduate',]]))
])

#4. Nominal Features
nominal_pipeline = Pipeline(steps=[
    ('encode', OneHotEncoder(handle_unknown="ignore",drop='first'))
])


preprocessor = ColumnTransformer(transformers=[
    ("Skewed_Pipeline", skew_pipeline, skew_num),
    ("Plain_Numeric",plain_numeric_pipeline, other_num ),
    ('Ordinal', ordinal_pipeline, ord_cat),
    ('Normal', nominal_pipeline, ohe_cat)
])


In [10]:
best_svr = Pipeline([['preprocess',preprocessor],['svr',SVR(C=10,epsilon=0.1,gamma=0.1)]])

In [11]:
tree_pipe = Pipeline([['preprocessor',preprocessor],['tree',RandomForestRegressor()]])

In [12]:
vote = VotingRegressor(estimators=[('svr',best_svr),('rf',tree_pipe)])

In [13]:
vote.fit(X,y)

,estimators,"[('svr', ...), ('rf', ...)]"
,weights,None
,n_jobs,None
,verbose,False
,transformers,"[('Skewed_Pipeline', ...), ('Plain_Numeric', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [14]:
import pickle 
with open("model.pkl", "wb") as f:
    pickle.dump(vote,f)

In [16]:
new_student = pd.DataFrame([[
    4.0,              # Study_Hours
    21,               # Age
    4.5,               # Avg_Daily_Usage_Hours
    134,               # Daily_Unlocks
    2.2,               # Physical_Activity_Hours
    6.7,               # Sleep_Hours_Per_Night
    'Medium',         # Stress_Level
    'Undergraduate',  # Academic_Level
    'Male',            # Gender
    'Other',           # Country
    'Facebook',        # Most_Used_Platform
    'Networking'       # Purpose_Of_Use
]], columns=cols)

In [17]:
new_student

,Study_Hours,Age,Avg_Daily_Usage_Hours,Daily_Unlocks,Physical_Activity_Hours,Sleep_Hours_Per_Night,Stress_Level,Academic_Level,Gender,Country,Most_Used_Platform,Purpose_Of_Use
0,4.0,21,4.5,134,2.2,6.7,Medium,Undergraduate,Male,Other,Facebook,Networking


In [18]:
vote.predict(new_student)

array([6.49903306])